# Module 5: RAG Pipeline with Azure DocumentDB (Python completed reference)

In [ ]:
from pymongo import MongoClientimport osconnection_string=os.environ.get("DOCUMENTDB_CONNECTION_STRING")if not connection_string: raise RuntimeError("Set DOCUMENTDB_CONNECTION_STRING")client=MongoClient(connection_string)db=client["docdbworkshop"]chunks=db["rag_chunks"]print(db.command({"ping":1}))

In [ ]:
chunks.drop()chunks.insert_many([ {"_id":"rag-001","sourceId":"search-module","title":"Vector search","chunk":"Azure DocumentDB vector search uses the $search stage with the cosmosSearch operator to retrieve documents by embedding similarity.","url":"module-4-search","tags":["vector","search"],"embedding":[0.92,0.80,0.18]}, {"_id":"rag-002","sourceId":"search-module","title":"Full-text search","chunk":"Azure DocumentDB full-text search uses createSearchIndexes and the $search text operator to return BM25-ranked keyword matches.","url":"module-4-search","tags":["full-text","bm25"],"embedding":[0.20,0.12,0.94]}, {"_id":"rag-003","sourceId":"search-module","title":"Hybrid search","chunk":"Hybrid search runs BM25 keyword retrieval and vector retrieval, then combines ranked lists with Reciprocal Rank Fusion.","url":"module-4-search","tags":["hybrid","rrf"],"embedding":[0.76,0.70,0.42]}, {"_id":"rag-004","sourceId":"rag-module","title":"Grounded generation","chunk":"A RAG pipeline retrieves relevant chunks from Azure DocumentDB and includes them in the model prompt so the answer is grounded in current application data.","url":"module-5-rag","tags":["rag","generation"],"embedding":[0.84,0.73,0.34]}])chunks.count_documents({})

In [ ]:
db.command({"createIndexes":"rag_chunks","indexes":[{"name":"idx_chunk_embedding_diskann","key":{"embedding":"cosmosSearch"},"cosmosSearchOptions":{"kind":"vector-diskann","dimensions":3,"similarity":"COS","maxDegree":32,"lBuild":64}}]})db.command({"createSearchIndexes":"rag_chunks","indexes":[{"name":"idx_chunk_fts","definition":{"mappings":{"dynamic":False,"fields":{"chunk":{"type":"string"}}}}}]})

In [ ]:
question="How does DocumentDB retrieve context for RAG?"question_vector=[0.83,0.74,0.33]vector_context=list(chunks.aggregate([{ "$search":{"cosmosSearch":{"path":"embedding","vector":question_vector,"k":3}}},{"$project":{"_id":1,"title":1,"chunk":1,"url":1,"score":{"$meta":"searchScore"}}}]))vector_context

In [ ]:
keyword_context=list(chunks.aggregate([{ "$search":{"index":"idx_chunk_fts","text":{"query":question,"path":"chunk"}}},{"$limit":3},{"$project":{"_id":1,"title":1,"chunk":1,"url":1,"score":{"$meta":"searchScore"}}}]))def rrf(lists,k=60,top_n=3):    docs={}; scores={}    for results in lists:        for rank,doc in enumerate(results):            i=str(doc["_id"]); docs[i]=doc; scores[i]=scores.get(i,0)+1/(k+rank+1)    return [{**docs[i],"rrfScore":s} for i,s in sorted(scores.items(),key=lambda x:x[1],reverse=True)[:top_n]]hybrid_context=rrf([keyword_context,vector_context])hybrid_context

In [ ]:
context_block="\n\n".join([f"[{i+1}] {d['title']}\n{d['chunk']}\nSource: {d['url']}" for i,d in enumerate(hybrid_context)])grounded_prompt=f"""You are a helpful assistant for an Azure DocumentDB workshop.Answer the user's question using only the context below. If the answer is not present, say you do not know.<context>{context_block}</context>Question: {question}"""print(grounded_prompt)